# Avito: Я использовал только TF-IDF


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

## Данные

In [ ]:
query_cols = [
    "search_query",
    "search_location_id",
    "search_is_delivery_search",
    "search_infm_params_text",
    "search_category",
]

item_cols = [
    "item_id",
    "item_title_raw",
    "item_description_raw",
    "item_infm_params_text",
]

data = pd.read_parquet("DATA/train.parquet", columns=query_cols + item_cols)
queries = pd.read_parquet("DATA/benchmark_queries.parquet")
items = pd.read_parquet("DATA/benchmark_items.parquet", columns=item_cols)

## Разбиение по запросам



In [ ]:
data["query_key"] = list(data[query_cols].itertuples(index=False, name=None))

train_keys, test_keys = train_test_split(
    data["query_key"].drop_duplicates(),
    test_size=0.3,
    random_state=42,
)

train_data = data[data["query_key"].isin(train_keys)]
test_data = data[data["query_key"].isin(test_keys)]

val_queries = (
    test_data[["query_key"] + query_cols]
    .drop_duplicates("query_key")
    .sample(n=1000, random_state=42)
    .reset_index(drop=True)
)

y_val = test_data.groupby("query_key")["item_id"].agg(set)
y_val = y_val.reindex(val_queries["query_key"])

## Корпус объявлений

Объединяем train и benchmark_items

In [ ]:
corpus = (
    pd.concat([items, data[item_cols]], ignore_index=True)
    .drop_duplicates("item_id")
    .reset_index(drop=True)
)

texts = (
    corpus["item_title_raw"].fillna("") + " "
    + corpus["item_description_raw"].fillna("") + " "
    + corpus["item_infm_params_text"].fillna("")
)
texts = texts.str.lower().str.replace("ё", "е", regex=False)

## TF-IDF

In [ ]:
vectorizer = TfidfVectorizer(
    max_features=100_000,
    min_df=2,
    dtype=np.float32,
)

item_vectors = vectorizer.fit_transform(texts)
item_ids = corpus["item_id"].to_numpy()

## Поиск top-50



In [ ]:
def predict_top50(query_texts, vectors, ids):
    query_texts = query_texts.fillna("").str.lower().str.replace("ё", "е", regex=False)
    query_vectors = vectorizer.transform(query_texts)
    predictions = []
    top_k = min(50, len(ids))

    for start in range(0, len(query_texts), 32):
        scores = (query_vectors[start:start + 32] @ vectors.T).toarray()

        top_indices = np.argpartition(-scores, top_k - 1, axis=1)[:, :top_k]
        for indices in top_indices:
            predictions.append(ids[indices].tolist())

    return predictions

## Recall@50

Локальная оценка на 1000 запросах

In [ ]:
val_predictions = predict_top50(val_queries["search_query"], item_vectors, item_ids)

recalls = [
    len(set(predicted) & correct) / len(correct)
    for predicted, correct in zip(val_predictions, y_val)
]

print(f"Recall@50: {np.mean(recalls):.4f}")

val_results = val_queries[["search_query", "search_location_id"]].copy()
val_results["correct_count"] = [len(correct) for correct in y_val]
val_results["recall@50"] = recalls
val_results.head(10)

## Ответ для бенчмарка

In [ ]:
benchmark_mask = corpus["item_id"].isin(items["item_id"]).to_numpy()
benchmark_vectors = item_vectors[benchmark_mask]
benchmark_ids = item_ids[benchmark_mask]

predictions = predict_top50(queries["search_query"], benchmark_vectors, benchmark_ids)

answer = pd.DataFrame({
    "query_id": queries["query_id"],
    "answer": [" ".join(ids) for ids in predictions],
})

answer.to_csv("answer.csv", index=False, encoding="utf-8")
answer.head()